<a href="https://colab.research.google.com/github/VeenusDadri/training/blob/master/advance_pyspark/data%20aggregation%20and%20join.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import os

In [6]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('Spark Training').getOrCreate()
sc = spark.sparkContext

In [7]:
print(os.path.exists("/content/population.csv"))

True


In [8]:
df = spark.read.format('csv').options(delimiter=',', header=True).load('/content/population.csv')

In [9]:
df.printSchema()

root
 |-- Country Name: string (nullable = true)
 |-- Country Code: string (nullable = true)
 |-- Year: string (nullable = true)
 |-- Value: string (nullable = true)



In [10]:
df.count()

16930

In [11]:
df.show(5, truncate=False)

+------------+------------+----+-----+
|Country Name|Country Code|Year|Value|
+------------+------------+----+-----+
|Aruba       |ABW         |1960|54922|
|Aruba       |ABW         |1961|55578|
|Aruba       |ABW         |1962|56320|
|Aruba       |ABW         |1963|57002|
|Aruba       |ABW         |1964|57619|
+------------+------------+----+-----+
only showing top 5 rows



In [12]:
df.show(5)

+------------+------------+----+-----+
|Country Name|Country Code|Year|Value|
+------------+------------+----+-----+
|       Aruba|         ABW|1960|54922|
|       Aruba|         ABW|1961|55578|
|       Aruba|         ABW|1962|56320|
|       Aruba|         ABW|1963|57002|
|       Aruba|         ABW|1964|57619|
+------------+------------+----+-----+
only showing top 5 rows



In [13]:
df1=df.dropDuplicates()


In [14]:
df1.count()

16930

In [15]:
df.select("Country Code").distinct().count()

265

In [16]:
df.select("Country Code").distinct().show()

+------------+
|Country Code|
+------------+
|         HTI|
|         PSE|
|         BRB|
|         LTE|
|         LVA|
|         POL|
|         ECS|
|         JAM|
|         TEA|
|         ZMB|
|         BRA|
|         MIC|
|         ARM|
|         IDA|
|         MOZ|
|         CUB|
|         JOR|
|         ABW|
|         FRA|
|         OSS|
+------------+
only showing top 20 rows



In [17]:
from pyspark.sql.functions import col,desc
df.filter((col('Country Name') == 'India') & (col('Year').isin('2015','2016'))).show()

+------------+------------+----+----------+
|Country Name|Country Code|Year|     Value|
+------------+------------+----+----------+
|       India|         IND|2015|1328024498|
|       India|         IND|2016|1343944296|
+------------+------------+----+----------+



In [18]:
df.select("Country Name").orderBy('Country Name', ascending=False).distinct().show(truncate=False)

+--------------------------------------------+
|Country Name                                |
+--------------------------------------------+
|South Asia                                  |
|Chad                                        |
|Lower middle income                         |
|Paraguay                                    |
|Low & middle income                         |
|Heavily indebted poor countries (HIPC)      |
|World                                       |
|Congo, Dem. Rep.                            |
|Senegal                                     |
|Cabo Verde                                  |
|Sweden                                      |
|East Asia & Pacific (IDA & IBRD countries)  |
|Kiribati                                    |
|Least developed countries: UN classification|
|Guyana                                      |
|Eritrea                                     |
|Philippines                                 |
|Pacific island small states                 |
|Djibouti    

In [19]:
df.select("Country Name").orderBy(col('Country Name').desc()).distinct().show(truncate=False)

+--------------------------------------------+
|Country Name                                |
+--------------------------------------------+
|South Asia                                  |
|Chad                                        |
|Lower middle income                         |
|Paraguay                                    |
|Low & middle income                         |
|Heavily indebted poor countries (HIPC)      |
|World                                       |
|Congo, Dem. Rep.                            |
|Senegal                                     |
|Cabo Verde                                  |
|Sweden                                      |
|East Asia & Pacific (IDA & IBRD countries)  |
|Kiribati                                    |
|Least developed countries: UN classification|
|Guyana                                      |
|Eritrea                                     |
|Philippines                                 |
|Pacific island small states                 |
|Djibouti    

In [20]:
df.select("Year").distinct().count()


64

In [21]:
df.select('Year').distinct().groupBy().count().show()

+-----+
|count|
+-----+
|   64|
+-----+



In [22]:
df.filter(df["Year"]==1990).agg({"Value" : "sum"}).show()

+---------------+
|     sum(Value)|
+---------------+
|5.5263399385E10|
+---------------+



In [23]:
df.filter(col('Year') == '1990').agg({'Value':'sum'}).show(truncate=False)

+---------------+
|sum(Value)     |
+---------------+
|5.5263399385E10|
+---------------+



In [24]:
df.filter(df.Year == '2007').agg({'Value':'min'}).show()

+----------+
|min(Value)|
+----------+
|    100150|
+----------+



In [25]:
df.filter(df.Value==df.filter(df.Year == '2007').agg({'Value':'min'}).collect()[0][0]).show()

+------------+------------+----+------+
|Country Name|Country Code|Year| Value|
+------------+------------+----+------+
|       Aruba|         ABW|2007|100150|
+------------+------------+----+------+



In [26]:
df1 = df.select('Country Name', 'Country Code').distinct()
df1.count()

265

In [38]:
df2 = df.select(col('Country Code').alias('ctry_cd'), 'Value', 'Year').distinct()
df2.count()

16930

In [29]:
df1.join(df2, col('Country Code') == col('ctry_cd')).count()

16930

In [31]:
df3=df1.join(df2, col('Country Code') == col('ctry_cd'))
df3.show(5)

+--------------------+------------+-------+---------+----+
|        Country Name|Country Code|ctry_cd|    Value|Year|
+--------------------+------------+-------+---------+----+
|               Aruba|         ABW|    ABW|    79805|1995|
|               Aruba|         ABW|    ABW|   100917|2008|
|         Afghanistan|         AFG|    AFG| 10036008|1965|
|Africa Western an...|         AFW|    AFW|440882906|2017|
| Antigua and Barbuda|         ATG|    ATG|    69612|1996|
+--------------------+------------+-------+---------+----+
only showing top 5 rows



In [34]:
df3.drop("ctry_cd").show(5)

+--------------------+------------+---------+----+
|        Country Name|Country Code|    Value|Year|
+--------------------+------------+---------+----+
|               Aruba|         ABW|    79805|1995|
|               Aruba|         ABW|   100917|2008|
|         Afghanistan|         AFG| 10036008|1965|
|Africa Western an...|         AFW|440882906|2017|
| Antigua and Barbuda|         ATG|    69612|1996|
+--------------------+------------+---------+----+
only showing top 5 rows



In [36]:
df3.drop("ctry_cd").show(5,False)

+--------------------------+------------+---------+----+
|Country Name              |Country Code|Value    |Year|
+--------------------------+------------+---------+----+
|Aruba                     |ABW         |79805    |1995|
|Aruba                     |ABW         |100917   |2008|
|Afghanistan               |AFG         |10036008 |1965|
|Africa Western and Central|AFW         |440882906|2017|
|Antigua and Barbuda       |ATG         |69612    |1996|
+--------------------------+------------+---------+----+
only showing top 5 rows



In [39]:

df2.limit(10).count()

10

In [40]:
df4 = df.orderBy('Country Code').limit(10)
df5 = df.orderBy('Country Code', ascending=False).limit(10)
df4.union(df5).show()

+------------+------------+----+-------+
|Country Name|Country Code|Year|  Value|
+------------+------------+----+-------+
|       Aruba|         ABW|1960|  54922|
|       Aruba|         ABW|1961|  55578|
|       Aruba|         ABW|1962|  56320|
|       Aruba|         ABW|1963|  57002|
|       Aruba|         ABW|1964|  57619|
|       Aruba|         ABW|1965|  58190|
|       Aruba|         ABW|1966|  58694|
|       Aruba|         ABW|1967|  58990|
|       Aruba|         ABW|1968|  59069|
|       Aruba|         ABW|1969|  59052|
|    Zimbabwe|         ZWE|1960|3809389|
|    Zimbabwe|         ZWE|1969|5058181|
|    Zimbabwe|         ZWE|1961|3930401|
|    Zimbabwe|         ZWE|1962|4055959|
|    Zimbabwe|         ZWE|1963|4185877|
|    Zimbabwe|         ZWE|1964|4320006|
|    Zimbabwe|         ZWE|1965|4458462|
|    Zimbabwe|         ZWE|1966|4601217|
|    Zimbabwe|         ZWE|1967|4748307|
|    Zimbabwe|         ZWE|1968|4900440|
+------------+------------+----+-------+



In [41]:
df.select(col('Country Name').alias('country'), col('Country Code').alias('code')).show(2)

+-------+----+
|country|code|
+-------+----+
|  Aruba| ABW|
|  Aruba| ABW|
+-------+----+
only showing top 2 rows



In [42]:
df.select('Year').distinct().groupBy().count().withColumnRenamed('count', 'year_count').show()

+----------+
|year_count|
+----------+
|        64|
+----------+



In [49]:
new_col_names = ['country','code','yr','val']
df6=df.toDF(*new_col_names)
df6.printSchema()

root
 |-- country: string (nullable = true)
 |-- code: string (nullable = true)
 |-- yr: string (nullable = true)
 |-- val: string (nullable = true)



In [50]:
new_col_names = ['country','code','yr','val']
df.toDF(*new_col_names).columns

['country', 'code', 'yr', 'val']

In [51]:
df.select(df['Value'].cast('bigint')).printSchema()

root
 |-- Value: long (nullable = true)



In [53]:
from pyspark.sql.functions import col
cols = df.columns #> ['Country Name', 'Country Code', 'Year', 'Value']
datatypes = ['string', 'string', 'bigint', 'bigint']
for i in range(len(cols)):
    df7 = df.withColumn(cols[i], col(cols[i]).cast(datatypes[i]))
df8 = df7.select(*cols)
df8.printSchema()

root
 |-- Country Name: string (nullable = true)
 |-- Country Code: string (nullable = true)
 |-- Year: long (nullable = true)
 |-- Value: long (nullable = true)



In [58]:
df.persist()

DataFrame[Country Name: string, Country Code: string, Year: bigint, Value: bigint]

In [62]:
df.storageLevel

StorageLevel(True, True, False, True, 1)

True condition indicates dataframe is present is already cached.

In [63]:
df.unpersist()

DataFrame[Country Name: string, Country Code: string, Year: bigint, Value: bigint]

In [64]:
df.storageLevel

StorageLevel(False, False, False, False, 1)